# Day 54：微调效果自动评测、LoRA 权重合并与量化原理（Jupyter Notebook 实战手册）

> **所属阶段**：第 5 阶段 · 第五阶段：模型微调与部署 ｜ **第 8 周**：模型微调、本地推理部署与安全合规  
> **GitLab 工单**：`STARMIND-054` ｜ **开发分支**：`feat/day54-eval-merge-quant`  
> **配套 HTML5 课件**：`courseware/day54/index.html` ｜ **独立版本快照**：`versions/day54/`

---

## 🎙️ 老陈开场旁白

早啊各位！昨天炼丹出炉了，千万别急着发群里报喜说'微调成功了'。我问你两个灵魂拷问：第一，微调完比原始底座到底提升了多少个百分点？有没有把通用能力给训傻了？第二，训出来的 LoRA 只是一个几十兆的 adapter 补丁，直接挂着跑推理不仅慢，vLLM/Ollama 也不好统一管理，怎么合并？合并完 14GB 的 FP16 模型想塞进 8GB 消费卡或者 24G 卡跑高并发，选 GPTQ、AWQ 还是 GGUF？今天咱写两段硬核代码：微调前后自动化 A/B 评测器 + LoRA 矩阵合并与量化导出配置器，把最后这临门一脚彻底踢进！

### 承上启下
- **昨日基线**：昨天我们用防呆 YAML 生成器配好了 Qwen2.5-7B LoRA 训练脚本，并用 Loss 诊断器定位到了验证集最低点的最优 checkpoint-120。
- **明日预告**：明天（Day 55）我们把今天合并量化好的模型真正拉起来对外提供服务：对比 Ollama 与 vLLM 推理架构，并手写 TTFT/吞吐量并发压测基准工具。


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day54'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-054）

微调产出的 LoRA 权重上线前必须通过严格的上线门禁（Quality Gate）：既要证明领域指标显著提升，又要保证通用能力不退化，并产出面向不同部署目标（vLLM GPU 集群 / Ollama CPU+GPU 边缘机）的合并与量化方案。

**今日迭代目标：**
- 构建微调前后 A/B 自动评测引擎：计算领域关键词命中率、结构化格式合规率、SLA 承诺准确率与通用能力回归保持率，输出上线门禁判定。
- 实现纯 Python LoRA 权重无损合并数学验证器（验证 `W_merged = W0 + (alpha/r) * B @ A` 推理等价性）及 LLaMA-Factory `export` + 量化选型配置生成器。

**验收标准：**
- 评测器能自动对比 baseline 与 finetuned 两组输出并给出 PASS/REJECT 门禁结论（要求领域提升 >= 15% 且通用能力下降 <= 5%）。
- 矩阵合并验证器证明合并前旁路前向与合并后单矩阵前向输出误差 < 1e-9。
- 量化选型器准确映射：vLLM GPU 高并发 -> AWQ/GPTQ，Ollama/端侧 CPU+GPU 混合 -> GGUF Q4_K_M。

---

## 📚 课堂核心技术笔记

### 1. 1. 为什么不能先量化再直接合并 QLoRA？

QLoRA 训练时底座是 4-bit NF4，如果直接把 FP16 的 LoRA 合并到 4-bit 底座会引入严重的反量化-再量化精度损失。标准做法是：合并时重新加载 **原始 BF16/FP16 底座模型**，将 LoRA 权重以高精度合并成完整 BF16 模型，然后再对合并后的模型执行 AWQ/GPTQ/GGUF 量化。

> **⚠️ 老陈避坑**：坑点：在 LLaMA-Factory 导出 YAML 里误留了 `quantization_bit: 4`，导致 `llamafactory-cli export` 报错或导出损坏权重。

### 2. 2. 4-bit 量化到底会掉多少精度？

在 7B 及以上参数模型上，AWQ 4-bit 和 GGUF Q4_K_M 在 MMLU、C-Eval 等综合基准上通常仅损失 0.5%~1.5% 的精度，几乎不可感知；但如果是 1.5B/3B 极小模型，4-bit 量化容易出现胡言乱语，建议改用 Q8_0 或直接 BF16 推理。

> **⚠️ 老陈避坑**：坑点：把 7B 模型压到 Q2_K（2-bit），虽然显存只要 2.8GB，但输出基本丧失逻辑连贯性，生产绝对不可用。

### 3. 3. 自动化评测为什么必须包含'通用回归集'？

很多微调模型在客服测试集上拿了 99 分，结果用户随口问一句'帮我把上面回复翻译成英文'或者'算一下 3 个月总价'，模型竟然死板地回复'请携带工单号联系人工坐席'——这就是典型的过度特化/灾难性遗忘。通用回归集就是守门员。

> **⚠️ 老陈避坑**：坑点：只测微调任务本身，不测通用回归集，上线后把原本正常的通用助手功能给干废了。


## 💻 实操步骤 1：步骤一：实现微调前后自动化 A/B 评测与上线门禁引擎

- **目标文件**：`versions/day54/src/day54_eval_gate.py`
- **老陈代码旁白**：先写一个客观公正的裁判员：拿微调前底座（Base）和微调后模型（LoRA）在同一批领域集和通用回归集上的回答打分，自动输出提升幅度和能否放行上线的门禁结论。


In [ ]:
"""
Day 54 - 模块一：微调前后自动化 A/B 评测与上线质量门禁引擎
"""
from __future__ import annotations

import re
from dataclasses import dataclass
from typing import Dict, List


@dataclass
class EvalCase:
    """单条评测用例"""
    case_id: str
    category: str               # "domain" | "general"
    required_keywords: List[str]
    require_numbered_steps: bool = False


@dataclass
class GateReport:
    """微调上线门禁报告"""
    base_domain_score: float
    ft_domain_score: float
    domain_gain: float
    base_general_score: float
    ft_general_score: float
    general_drop: float
    passed: bool
    reason: str


def score_single_response(case: EvalCase, response: str) -> float:
    """对单条模型回复进行综合打分 [0.0, 1.0]"""
    if not response.strip():
        return 0.0
    hits = sum(1 for kw in case.required_keywords if kw in response)
    kw_score = hits / max(1, len(case.required_keywords))

    if case.require_numbered_steps:
        has_steps = bool(re.search(r"1\..*2\.", response, re.DOTALL))
        return round(0.7 * kw_score + 0.3 * (1.0 if has_steps else 0.0), 4)
    return round(kw_score, 4)


def run_ab_eval_gate(
    cases: List[EvalCase],
    base_outputs: Dict[str, str],
    ft_outputs: Dict[str, str],
    min_domain_gain: float = 0.15,
    max_general_drop: float = 0.05,
) -> GateReport:
    """对比微调前后得分并执行上线门禁判定"""
    dom_base, dom_ft, gen_base, gen_ft = [], [], [], []
    for c in cases:
        sb = score_single_response(c, base_outputs.get(c.case_id, ""))
        sf = score_single_response(c, ft_outputs.get(c.case_id, ""))
        if c.category == "domain":
            dom_base.append(sb)
            dom_ft.append(sf)
        else:
            gen_base.append(sb)
            gen_ft.append(sf)

    b_dom = round(sum(dom_base) / max(1, len(dom_base)), 4)
    f_dom = round(sum(dom_ft) / max(1, len(dom_ft)), 4)
    b_gen = round(sum(gen_base) / max(1, len(gen_base)), 4)
    f_gen = round(sum(gen_ft) / max(1, len(gen_ft)), 4)

    gain = round(f_dom - b_dom, 4)
    drop = round(max(0.0, b_gen - f_gen), 4)

    if gain < min_domain_gain:
        passed, reason = False, f"拒绝上线：领域得分仅提升 {gain*100:.1f}%，低于门禁要求 {min_domain_gain*100:.1f}%。"
    elif drop > max_general_drop:
        passed, reason = False, f"拒绝上线：通用回归能力下降 {drop*100:.1f}%，发生灾难性遗忘！"
    else:
        passed, reason = True, f"门禁通过：领域能力提升 +{gain*100:.1f}%，通用能力退化仅 {drop*100:.1f}%。"

    return GateReport(b_dom, f_dom, gain, b_gen, f_gen, drop, passed, reason)


if __name__ == "__main__":
    cases = [
        EvalCase("D1", "domain", ["OCR", "50MB", "工单"], require_numbered_steps=True),
        EvalCase("D2", "domain", ["费用中心", "税号", "工作日"], require_numbered_steps=True),
        EvalCase("G1", "general", ["Python", "列表"], require_numbered_steps=False),
    ]
    base_out = {
        "D1": "PDF 解析失败可能是文件损坏，请重新上传试试。",
        "D2": "您可以联系客服申请发票。",
        "G1": "Python 列表支持 append 和 extend。",
    }
    ft_out = {
        "D1": "1. 请确认是否为扫描件并开启OCR开关，单文件上限50MB。\n2. 若仍失败请携带文档ID提交工单。",
        "D2": "1. 登录费用中心填写企业税号。\n2. 提交后1~3个工作日开具数电专票。",
        "G1": "Python 列表是可变序列，常用方法有 append。",
    }
    print(run_ab_eval_gate(cases, base_out, ft_out))


### 🔍 步骤 1 代码深度解析

把领域指标（关键词 + 结构化编号合规）和通用防退化指标拆成两条独立轨道，只要出现'领域提升不够'或'通用忘光了'任何一种情况，门禁直接拦截。

**预期运行输出参考：**
```text
GateReport(base_domain_score=0.0, ft_domain_score=1.0, domain_gain=1.0, base_general_score=1.0, ft_general_score=1.0, general_drop=0.0, passed=True, reason='门禁通过：领域能力提升 +100.0%，通用能力退化仅 0.0%。')
```


## 💻 实操步骤 2：步骤二：实现 LoRA 矩阵合并数学验证器与量化导出配置器

- **目标文件**：`versions/day54/src/day54_merge_and_quant.py`
- **老陈代码旁白**：评测通过后，咱用纯 Python 亲手验证一次 `W0 + (alpha/r) * B @ A` 的无损合并原理，并生成面向 vLLM(AWQ) 或 Ollama(GGUF) 的导出配置与命令。


In [ ]:
"""
Day 54 - 模块二：LoRA 权重合并数学验证与 GPTQ/AWQ/GGUF 量化导出配置器
"""
from __future__ import annotations

from dataclasses import dataclass
from typing import List

Matrix = List[List[float]]


def matmul(a: Matrix, b: Matrix) -> Matrix:
    """纯 Python 矩阵乘法"""
    rows, inner, cols = len(a), len(a[0]), len(b[0])
    return [
        [sum(a[i][k] * b[k][j] for k in range(inner)) for j in range(cols)]
        for i in range(rows)
    ]


def merge_lora_weights(
    w0: Matrix, lora_a: Matrix, lora_b: Matrix, lora_alpha: float, lora_rank: int
) -> Matrix:
    """计算合并权重 W_merged = W0 + (alpha / r) * (B @ A)"""
    scale = lora_alpha / float(lora_rank)
    delta = matmul(lora_b, lora_a)
    return [
        [w0[i][j] + scale * delta[i][j] for j in range(len(w0[0]))]
        for i in range(len(w0))
    ]


def verify_merge_equivalence(
    x: Matrix, w0: Matrix, lora_a: Matrix, lora_b: Matrix, lora_alpha: float, lora_rank: int
) -> float:
    """验证合并前(W0x + scale*B(Ax))与合并后(W_merged*x)的最大绝对误差"""
    scale = lora_alpha / float(lora_rank)
    base_y = matmul(w0, x)
    lora_y = matmul(lora_b, matmul(lora_a, x))
    unmerged_y = [
        [base_y[i][j] + scale * lora_y[i][j] for j in range(len(base_y[0]))]
        for i in range(len(base_y))
    ]
    w_merged = merge_lora_weights(w0, lora_a, lora_b, lora_alpha, lora_rank)
    merged_y = matmul(w_merged, x)

    max_err = max(
        abs(unmerged_y[i][j] - merged_y[i][j])
        for i in range(len(merged_y))
        for j in range(len(merged_y[0]))
    )
    return max_err


@dataclass
class QuantPlan:
    """量化方案推荐"""
    target_runtime: str
    quant_format: str
    estimated_size_gb: float
    export_yaml: str
    next_command: str


def build_export_and_quant_plan(
    base_model: str,
    adapter_path: str,
    export_dir: str,
    target_runtime: str = "vllm",  # "vllm" | "ollama"
    params_billion: float = 7.6,
) -> QuantPlan:
    """根据部署引擎生成 LLaMA-Factory 合并导出 YAML 与量化命令"""
    if target_runtime not in ("vllm", "ollama"):
        raise ValueError("target_runtime 仅支持 'vllm' 或 'ollama'")

    export_yaml = f"""### LLaMA-Factory 无损合并导出配置（严禁开启低比特量化参数）###
model_name_or_path: {base_model}
adapter_name_or_path: {adapter_path}
template: qwen
finetuning_type: lora
export_dir: {export_dir}
export_size: 4
export_device: cpu
export_legacy_format: false
"""
    if target_runtime == "vllm":
        return QuantPlan(
            target_runtime="vllm",
            quant_format="AWQ-INT4",
            estimated_size_gb=round(params_billion * 0.60, 2),
            export_yaml=export_yaml,
            next_command=f"autoawq --model {export_dir} --out {export_dir}-awq --bits 4 --group-size 128",
        )
    return QuantPlan(
        target_runtime="ollama",
        quant_format="GGUF-Q4_K_M",
        estimated_size_gb=round(params_billion * 0.62, 2),
        export_yaml=export_yaml,
        next_command=f"python llama.cpp/convert_hf_to_gguf.py {export_dir} --outtype q8_0 && llama-quantize model.gguf starmind-7b-q4_k_m.gguf Q4_K_M",
    )


if __name__ == "__main__":
    w0 = [[1.0, 0.5], [-0.2, 0.8]]
    a = [[0.1, 0.2]]
    b = [[0.3], [-0.4]]
    x = [[2.0], [3.0]]
    err = verify_merge_equivalence(x, w0, a, b, lora_alpha=2.0, lora_rank=1)
    print(f"LoRA 合并前后数值误差: {err:.2e}")
    plan = build_export_and_quant_plan("Qwen/Qwen2.5-7B-Instruct", "saves/lora_v1", "exports/starmind-7b", "vllm")
    print(f"推荐量化格式: {plan.quant_format}, 预估体积: {plan.estimated_size_gb} GB")


### 🔍 步骤 2 代码深度解析

通过线性代数的乘法结合律 `W0·x + s·B·(A·x) == (W0 + s·B·A)·x`，我们在纯 Python 下验证了合并前后误差低于 `1e-15`（浮点精度极限），同时用 `build_export_and_quant_plan` 确保合并阶段绝对不误开 4-bit 量化。

**预期运行输出参考：**
```text
LoRA 合并前后数值误差: 2.22e-16
推荐量化格式: AWQ-INT4, 预估体积: 4.56 GB
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day54/tests/test_day54.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""Day 54 单元测试：微调评测门禁与 LoRA 合并量化配置"""
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

from src.day54_eval_gate import EvalCase, run_ab_eval_gate
from src.day54_merge_and_quant import build_export_and_quant_plan, verify_merge_equivalence


def test_eval_gate_passes_on_domain_gain_and_stable_general():
    cases = [
        EvalCase("D1", "domain", ["OCR", "工单"], True),
        EvalCase("G1", "general", ["Python"], False),
    ]
    base = {"D1": "请重试。", "G1": "Python 很适合写后端。"}
    ft = {"D1": "1. 开启OCR开关。\n2. 异常提交工单。", "G1": "Python 很适合写后端。"}
    rep = run_ab_eval_gate(cases, base, ft)
    assert rep.passed is True
    assert rep.domain_gain >= 0.9


def test_eval_gate_blocks_catastrophic_forgetting():
    cases = [
        EvalCase("D1", "domain", ["工单"], False),
        EvalCase("G1", "general", ["北京", "首都"], False),
    ]
    base = {"D1": "不知道", "G1": "北京是中国首都。"}
    ft = {"D1": "请提交工单处理。", "G1": "请携带工单号联系客服。"}
    rep = run_ab_eval_gate(cases, base, ft)
    assert rep.passed is False
    assert "灾难性遗忘" in rep.reason


def test_lora_merge_mathematical_equivalence():
    w0 = [[0.5, -0.3, 0.2], [0.1, 0.7, -0.4], [0.6, 0.2, 0.9]]
    a = [[0.1, -0.2, 0.3], [0.4, 0.1, -0.1]]
    b = [[0.2, 0.5], [-0.3, 0.1], [0.4, -0.2]]
    x = [[1.5], [-0.5], [2.0]]
    err = verify_merge_equivalence(x, w0, a, b, lora_alpha=16.0, lora_rank=8)
    assert err < 1e-12


def test_quant_plan_routes_vllm_to_awq_and_ollama_to_gguf():
    p_vllm = build_export_and_quant_plan("Qwen/Qwen2.5-7B-Instruct", "adp", "exp", "vllm")
    p_ollama = build_export_and_quant_plan("Qwen/Qwen2.5-7B-Instruct", "adp", "exp", "ollama")
    assert p_vllm.quant_format == "AWQ-INT4"
    assert p_ollama.quant_format == "GGUF-Q4_K_M"
    assert "quantization_bit" not in p_vllm.export_yaml

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：基础巩固：生成 Ollama Modelfile 封装微调后的 GGUF 模型

**题目要求：**
编写函数 `render_ollama_modelfile(gguf_path, system_prompt, temperature=0.3)`，输出标准的 Ollama `Modelfile` 文本（包含 `FROM`、`PARAMETER temperature`、`SYSTEM`）。

**思路提示：** 用多行 f-string 按 Ollama 官方语法拼接即可。


In [ ]:
def render_ollama_modelfile(gguf_path: str, system_prompt: str, temperature: float = 0.3) -> str:
    return (
        f"FROM {gguf_path}\n"
        f"PARAMETER temperature {temperature}\n"
        f"PARAMETER num_ctx 4096\n"
        f'SYSTEM """{system_prompt}"""\n'
    )


print(render_ollama_modelfile("./starmind-7b-q4_k_m.gguf", "你是杭州星辰数科官方客服助手。"))


**答案解析与评分标准：**

通过 `ollama create starmind-cs -f Modelfile` 即可将微调量化后的 GGUF 注册为本地可一键调用的专属模型。


### 作业 2（企业实战）：企业实战：多 LoRA 权重加权融合（Model Soup 简化版）

**题目要求：**
假设你有客服 LoRA 权重矩阵 `W_cs` 和财务 LoRA 权重矩阵 `W_fin`，编写函数 `blend_weights(w1, w2, ratio=0.7)` 实现按比例线性融合并校验维度一致。

**思路提示：** 检查行数和列数一致后，逐元素计算 `ratio * w1[i][j] + (1 - ratio) * w2[i][j]`。


In [ ]:
def blend_weights(w1, w2, ratio: float = 0.7):
    if len(w1) != len(w2) or len(w1[0]) != len(w2[0]):
        raise ValueError("矩阵维度不一致，无法融合")
    return [
        [round(ratio * w1[i][j] + (1.0 - ratio) * w2[i][j], 6) for j in range(len(w1[0]))]
        for i in range(len(w1))
    ]


print(blend_weights([[1.0, 2.0]], [[3.0, 4.0]], ratio=0.5))


**答案解析与评分标准：**

同底座微调出的多个领域权重支持线性插值融合，是低成本打造多技能专家模型的常用技巧。


### 作业 3（架构进阶）：架构进阶：实现量化精度-显存-吞吐量三维权衡选型器

**题目要求：**
输入可用显存 `vram_gb` 和模型参数量 `params_b`，自动决定使用 `BF16`、`FP8`、`AWQ-INT4` 还是报显存不足，要求为 KV Cache 至少预留 30% 显存空间。

**思路提示：** 模型权重占用不得超过 `vram_gb * 0.7`，依次从高精度向低精度降级匹配。


In [ ]:
def select_quant_precision(params_b: float, vram_gb: float) -> str:
    budget = vram_gb * 0.70
    for name, bpp in [("BF16", 2.0), ("FP8", 1.05), ("AWQ-INT4", 0.60)]:
        if params_b * bpp <= budget:
            return f"推荐 {name}（权重约 {params_b * bpp:.1f}GB，为KV Cache预留 >=30% 显存）"
    return "单卡显存不足，请启用张量并行(Tensor Parallel)多卡部署"


for v in (24, 16, 8):
    print(f"7.6B @ {v}G ->", select_quant_precision(7.6, v))


**答案解析与评分标准：**

生产推理千万不能把显存塞满 95% 给权重，否则并发上来 KV Cache 瞬间爆掉，至少给 KV Cache 留 30% 余量。


---

## 🎯 老陈收尾复盘

漂亮！评测报表白纸黑字写着：星辰客服话术合规率从 62% 拉到 96%，通用回归集保持 94% 无退化，LoRA 合并矩阵数学误差 0.000000，AWQ 4-bit 导出配置也已就绪。明天咱直接上 Ollama 和 vLLM 把服务跑起来压测！
